In [10]:
# Install libraries
!pip install transformers torch

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load model
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

print("🤖 Chatbot Ready! Type 'exit' to stop.\n")

chat_history_ids = None

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! 👋")
        break

    # ✅ Step 1: Add polite instruction
    user_input = "Answer politely and professionally: " + user_input

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Add chat history
    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    # ✅ Step 2: Better response generation
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.6,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode response
    bot_reply = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    # ✅ Step 3: Filter bad/weird responses
    bad_words = ["dad", "stupid", "hate"]

    if any(word in bot_reply.lower() for word in bad_words):
        bot_reply = "I am an AI chatbot created to assist you politely."

    print("Chatbot:", bot_reply)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🤖 Chatbot Ready! Type 'exit' to stop.

You:  What is your name?
Chatbot: I'm not sure if I should upvote or down vote.
You: Helo
Chatbot: My phone is dead, but you're the best!
You: Hello
Chatbot: Hello, hello?
You: How are you?
Chatbot: How's it going?
You: exit
Chatbot: Goodbye! 👋
